# Five 4K60 streams made the deadline. Six did not.

This notebook explores the published evidence from a paced NVDEC → CUDA processing → NVENC study. The question is not how quickly a file can be transcoded. It is whether every arriving frame finishes inside its live frame budget without an ever-growing queue.

Real uses include multi-camera analytics, live overlays, vision inference pre-processing, remote production, transcoding gateways, and recorders that must process several feeds on one GPU.


In [1]:
from pathlib import Path
import csv, html, json
from IPython.display import HTML, display

def show_table(headers, body):
    head = "".join(f"<th>{html.escape(str(value))}</th>" for value in headers)
    rows_html = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row) + "</tr>"
        for row in body
    )
    display(HTML(
        '<div style="overflow-x:auto"><table style="border-collapse:collapse;min-width:620px">'
        f"<thead><tr>{head}</tr></thead><tbody>{rows_html}</tbody></table></div>"
    ))

def capacity_chart(capacity):
    width, height = 820, 430
    left, right, top, bottom, maximum = 72, 24, 34, 68, 500
    chart_w, chart_h = width - left - right, height - top - bottom
    group_w = chart_w / len(capacity)
    parts = [
        f'<svg viewBox="0 0 {width} {height}" width="100%" style="max-width:820px;background:#fff">',
        '<style>.axis{stroke:#334155}.grid{stroke:#cbd5e1}.label{font:13px system-ui;fill:#334155}</style>'
    ]
    for tick in range(0, maximum + 1, 100):
        y = top + chart_h - tick / maximum * chart_h
        parts += [f'<line x1="{left}" y1="{y}" x2="{width-right}" y2="{y}" class="grid"/>',
                  f'<text x="{left-10}" y="{y+4}" text-anchor="end" class="label">{tick}</text>']
    for index, row in enumerate(capacity):
        center = left + group_w * (index + 0.5)
        target, achieved = float(row["target_fps_total"]), float(row["achieved_fps_median"])
        target_h, achieved_h = target / maximum * chart_h, achieved / maximum * chart_h
        color = "#16856b" if int(row["sla_passes"]) == 3 else "#d97706"
        parts += [
            f'<rect x="{center-24}" y="{top+chart_h-target_h}" width="48" height="{target_h}" fill="none" stroke="#64748b" stroke-width="2" stroke-dasharray="5 4"/>',
            f'<rect x="{center-18}" y="{top+chart_h-achieved_h}" width="36" height="{achieved_h}" fill="{color}"/>',
            f'<text x="{center}" y="{top+chart_h+23}" text-anchor="middle" class="label">{row["streams"]}</text>',
            f'<text x="{center}" y="{top+chart_h-achieved_h-7}" text-anchor="middle" class="label">{achieved:.0f}</text>'
        ]
    parts += [f'<line x1="{left}" y1="{top+chart_h}" x2="{width-right}" y2="{top+chart_h}" class="axis"/>', '</svg>']
    return "".join(parts)

root = Path.cwd()
if not (root / "results").exists():
    root = root / "realtime-gpu-video"

with (root / "results" / "case-medians.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))
summary = json.loads((root / "results" / "summary.json").read_text())

print(f"{len(rows)} confirmed cases, {summary['run_count']} timed runs")
print(summary["capacity_boundary"])


17 confirmed cases, 51 timed runs
{'largest_confirmed_passing_stream_count': 5, 'saturated_fps_median': 333.56790628867395, 'smallest_confirmed_failing_stream_count': 6}


## The SLA

At 60 FPS, a frame arrives every **16.667 ms**. A condition passed only when all three replications met all three rules:

- aggregate throughput reached at least 99% of the arrival rate;
- no more than 1% of measured frames missed one frame period; and
- p99 arrival-to-completion latency stayed within one frame period.

Each run also had to match a NumPy processing oracle exactly and produce a bitstream that FFmpeg could independently decode and count.


## The concurrency boundary


In [2]:
capacity = [
    row for row in rows
    if row["width"] == "3840"
    and row["target_fps_per_stream"] == "60"
    and row["effect"] == "pointwise"
    and row["placement"] == "device"
    and row["codec"] == "h264"
]
capacity.sort(key=lambda row: int(row["streams"]))

for row in capacity:
    print(
        f'{row["streams"]} streams: target {row["target_fps_total"]} FPS, '
        f'achieved {float(row["achieved_fps_median"]):.1f}, '
        f'p99 {float(row["latency_p99_ms_median"]):.2f} ms, '
        f'{row["sla_passes"]}/3 passes'
    )
show_table(
    ["Streams", "Target FPS", "Achieved FPS", "p99 latency", "Miss rate", "SLA"],
    [[row["streams"], row["target_fps_total"], f'{float(row["achieved_fps_median"]):.1f}',
      f'{float(row["latency_p99_ms_median"]):.2f} ms',
      f'{100*float(row["deadline_miss_rate_median"]):.2f}%',
      "pass" if int(row["sla_passes"]) == 3 else "fail"] for row in capacity]
)
display(HTML(capacity_chart(capacity)))


1 streams: target 60 FPS, achieved 60.1, p99 1.00 ms, 3/3 passes
2 streams: target 120 FPS, achieved 120.2, p99 1.26 ms, 3/3 passes
4 streams: target 240 FPS, achieved 240.4, p99 4.24 ms, 3/3 passes
5 streams: target 300 FPS, achieved 300.5, p99 2.16 ms, 3/3 passes
6 streams: target 360 FPS, achieved 333.4, p99 777.94 ms, 0/3 passes
8 streams: target 480 FPS, achieved 333.7, p99 4238.01 ms, 0/3 passes


Streams,Target FPS,Achieved FPS,p99 latency,Miss rate,SLA
1,60,60.1,1.00 ms,0.34%,pass
2,120,120.2,1.26 ms,0.34%,pass
4,240,240.4,4.24 ms,0.56%,pass
5,300,300.5,2.16 ms,0.31%,pass
6,360,333.4,777.94 ms,95.64%,fail
8,480,333.7,4238.01 ms,98.50%,fail


Five streams delivered the required 300 FPS in all three replications. Six requested 360 FPS, but the pipeline flattened near 333 FPS. Eight streams requested more work without raising the completed-frame rate.

## Why a small deficit becomes seconds of latency


In [3]:
capacity_fps = summary["capacity_boundary"]["saturated_fps_median"]
duration_seconds = 10
for arrival_fps in (300, 360, 480):
    backlog_frames = max(0.0, arrival_fps - capacity_fps) * duration_seconds
    latency_debt_seconds = backlog_frames / capacity_fps
    print(
        f"{arrival_fps:>3} FPS arrival -> {backlog_frames:>7.1f} queued frames, "
        f"about {latency_debt_seconds:>4.2f} s of debt after 10 s"
    )


300 FPS arrival ->     0.0 queued frames, about 0.00 s of debt after 10 s
360 FPS arrival ->   264.3 queued frames, about 0.79 s of debt after 10 s
480 FPS arrival ->  1464.3 queued frames, about 4.39 s of debt after 10 s


The simple capacity model closely matches the measured direction. At 360 FPS, a roughly 26 FPS deficit builds about 0.8 seconds of debt over ten seconds. At 480 FPS, it builds more than four seconds. A live pipeline cannot average that debt away; it must buffer, drop, or reduce work.

## Placement spent the budget before the codec engines were full


In [4]:
placement = [
    row for row in rows
    if row["width"] == "3840"
    and row["target_fps_per_stream"] == "60"
    and row["streams"] == "1"
    and row["codec"] == "h264"
    and (row["placement"] == "host_roundtrip" or row["effect"] in {"passthrough", "pointwise", "unsharp", "stacked"})
]
placement.sort(key=lambda row: (row["placement"], row["effect"]))
for row in placement:
    print(
        f'{row["placement"]:>14} / {row["effect"]:<11}: '
        f'{float(row["achieved_fps_median"]):5.1f} FPS, '
        f'p99 {float(row["latency_p99_ms_median"]):7.2f} ms, '
        f'{row["sla_passes"]}/3 passes'
    )
show_table(
    ["Placement", "Effect", "Achieved FPS", "p99 latency", "Miss rate", "SLA passes"],
    [[row["placement"], row["effect"], f'{float(row["achieved_fps_median"]):.1f}',
      f'{float(row["latency_p99_ms_median"]):.2f} ms',
      f'{100*float(row["deadline_miss_rate_median"]):.1f}%', f'{row["sla_passes"]}/3']
     for row in placement]
)


        device / passthrough:  60.1 FPS, p99    1.04 ms, 3/3 passes
        device / pointwise  :  60.1 FPS, p99    1.00 ms, 3/3 passes
        device / stacked    :  60.1 FPS, p99    1.11 ms, 3/3 passes
        device / unsharp    :  60.1 FPS, p99    0.93 ms, 3/3 passes
host_roundtrip / passthrough:  60.1 FPS, p99    5.40 ms, 3/3 passes
host_roundtrip / pointwise  :  49.9 FPS, p99 1962.59 ms, 0/3 passes
host_roundtrip / unsharp    :  36.1 FPS, p99 6397.42 ms, 0/3 passes


Placement,Effect,Achieved FPS,p99 latency,Miss rate,SLA passes
device,passthrough,60.1,1.04 ms,0.3%,3/3
device,pointwise,60.1,1.00 ms,0.3%,3/3
device,stacked,60.1,1.11 ms,0.3%,3/3
device,unsharp,60.1,0.93 ms,0.2%,3/3
host_roundtrip,passthrough,60.1,5.40 ms,0.3%,3/3
host_roundtrip,pointwise,49.9,1962.59 ms,100.0%,0/3
host_roundtrip,unsharp,36.1,6397.42 ms,100.0%,0/3


A transfer-only roundtrip still passed, but its p99 rose from roughly one millisecond to about five. Adding NumPy pointwise work reduced throughput to about 50 FPS; the unsharp control fell near 36 FPS. The lesson is not that one copy always breaks 4K60. It is that copies and CPU work consume the same finite frame budget.

## Codec choice was not the single-stream limit


In [5]:
codecs = [
    row for row in rows
    if row["width"] == "3840"
    and row["target_fps_per_stream"] == "60"
    and row["streams"] == "1"
    and row["effect"] == "pointwise"
    and row["placement"] == "device"
]
for row in sorted(codecs, key=lambda row: row["codec"]):
    print(
        f'{row["codec"].upper():>4}: p99 {float(row["latency_p99_ms_median"]):.2f} ms, '
        f'NVENC mean {float(row["encoder_percent_mean_median"]):.1f}%, '
        f'output {float(row["output_mbps_median"]):.1f} Mb/s'
    )
show_table(
    ["Codec", "Achieved FPS", "p99 latency", "NVENC mean", "Output rate", "SLA"],
    [[row["codec"].upper(), f'{float(row["achieved_fps_median"]):.1f}',
      f'{float(row["latency_p99_ms_median"]):.2f} ms',
      f'{float(row["encoder_percent_mean_median"]):.1f}%',
      f'{float(row["output_mbps_median"]):.1f} Mb/s', f'{row["sla_passes"]}/3']
     for row in sorted(codecs, key=lambda row: row["codec"])]
)


 AV1: p99 0.96 ms, NVENC mean 14.4%, output 41.0 Mb/s
H264: p99 1.00 ms, NVENC mean 24.3%, output 39.8 Mb/s
HEVC: p99 0.98 ms, NVENC mean 13.5%, output 39.9 Mb/s


Codec,Achieved FPS,p99 latency,NVENC mean,Output rate,SLA
AV1,60.1,0.96 ms,14.4%,41.0 Mb/s,3/3
H264,60.1,1.00 ms,24.3%,39.8 Mb/s,3/3
HEVC,60.1,0.98 ms,13.5%,39.9 Mb/s,3/3


All three hardware codecs sustained one 4K60 stream with this low-latency P1 configuration. These output rates are not a compression-quality comparison: the study held its synthetic source and rate-control inputs fixed to study the service boundary, not perceptual quality.

## What this notebook does not claim

The source was a ten-second synthetic H.264 clip in warm local storage. The host ran ordinary Ubuntu, not a hard-real-time kernel. The effects touched NV12 luma, not a neural network or multi-frame model. The study did not include capture-card ingress, network jitter, audio, mux latency, B-frames, quality sweeps, long thermal soaks, or recovery after dropped frames.

The useful result is narrower: on this RTX PRO 4000 SFF and this decode/process/encode path, five concurrent 4K60 feeds were confirmed inside the SLA and six were not. Keeping the processing on the GPU preserved far more of the frame budget than returning each frame to NumPy.
